# 01 Exploratory Data Analysis (EDA)

This notebook explores the credit card fraud dataset, focusing on:
- Dataset overview and structural checks
- Outlier detection using the Interquartile Range (IQR) method
- Class imbalance and density distributions
- PCA feature correlations
- Dimensionality reduction and class separability using 2D PCA & t-SNE projections

Dataset source: Kaggle — Credit Card Fraud Detection  
Place the dataset at: `data/creditcard.csv`

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set(style="whitegrid")
os.makedirs("../reports", exist_ok=True)

## 1. Load Dataset & Structural Summary

In [ ]:
def load_dataset(path):
    data = pd.read_csv(path)
    print("\n=== DATASET OVERVIEW ===")
    print("Head:")
    print(data.head())
    print("Shape:", data.shape)
    print("\nMissing values per column:")
    print(data.isnull().sum())
    print("\nNumber of duplicated rows:", data.duplicated().sum())
    print("\nClass distribution:")
    print(data["Class"].value_counts())
    print("Fraud percentage:", data["Class"].mean() * 100)
    print("\nData types:")
    print(data.dtypes)
    return data

data_path = "../data/creditcard.csv"
data = load_dataset(data_path)

## 2. Create Data Dictionary

In [ ]:
def create_data_dictionary():
    data_dict = pd.DataFrame(
        [
            ["Time", "float", "Seconds since first transaction", "seconds", ">= 0"],
            ["Amount", "float", "Transaction amount", "currency units", ">= 0"],
        ]
        + [
            [f"V{i}", "float", "PCA-transformed anonymized feature", "N/A", "continuous"]
            for i in range(1, 29)
        ]
        + [
            ["Class", "int", "Target variable: 0 = non-fraud, 1 = fraud", "N/A", "{0,1}"]
        ],
        columns=["Variable", "Type", "Description", "Units", "Allowed Values"]
    )
    print("\n=== DATA DICTIONARY ===")
    print(data_dict)
    data_dict.to_csv("../reports/data_dictionary.csv", index=False)

create_data_dictionary()

## 3. Outlier Analysis & Exploratory Visualizations

In [ ]:
def check_amount_outliers(data):
    Q1 = data["Amount"].quantile(0.25)
    Q3 = data["Amount"].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_amount = ((data["Amount"] < lower_bound) | (data["Amount"] > upper_bound)).sum()
    print("\nOutliers in Amount (IQR rule):", outliers_amount)

def basic_plots(data):
    plt.figure(figsize=(5, 4))
    sns.countplot(x="Class", data=data)
    plt.title("Class Distribution (0 = Non-Fraud, 1 = Fraud)")
    plt.show()

    fraud = data[data["Class"] == 1]
    non_fraud = data[data["Class"] == 0]
    print("\nFraud transactions:", len(fraud))
    print("Non-fraud transactions:", len(non_fraud))
    print("Fraud percentage:", len(fraud) / len(data) * 100)

    plt.figure(figsize=(8, 5))
    sns.histplot(non_fraud["Amount"], bins=50, color="blue", label="Non-Fraud", stat="density", kde=True)
    sns.histplot(fraud["Amount"], bins=50, color="red", label="Fraud", stat="density", kde=True)
    plt.legend()
    plt.title("Transaction Amount Distribution by Class")
    plt.show()

    plt.figure(figsize=(12, 10))
    sns.heatmap(data.corr(), cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.show()

check_amount_outliers(data)
basic_plots(data)

## 4. Dimensionality Reduction (PCA & t-SNE)

In [ ]:
def pca_tsne_plots(X, y):
    pca = PCA(n_components=2, random_state=42)
    pca_sample = X.sample(min(50000, len(X)), random_state=42)
    y_pca_sample = y.loc[pca_sample.index]
    X_pca = pca.fit_transform(pca_sample)

    plt.figure(figsize=(8, 6))
    plt.scatter(X_pca[y_pca_sample == 0, 0], X_pca[y_pca_sample == 0, 1], s=5, alpha=0.5, label="Non-Fraud")
    plt.scatter(X_pca[y_pca_sample == 1, 0], X_pca[y_pca_sample == 1, 1], s=10, alpha=0.8, c="red", label="Fraud")
    plt.title("PCA Projection (Large Sample)")
    plt.legend()
    plt.show()

    tsne_sample = X.sample(min(10000, len(X)), random_state=42)
    y_tsne_sample = y.loc[tsne_sample.index]
    X_tsne = TSNE(n_components=2, random_state=42).fit_transform(tsne_sample)

    plt.figure(figsize=(8, 6))
    plt.scatter(X_tsne[y_tsne_sample == 0, 0], X_tsne[y_tsne_sample == 0, 1], s=5, alpha=0.5, label="Non-Fraud")
    plt.scatter(X_tsne[y_tsne_sample == 1, 0], X_tsne[y_tsne_sample == 1, 1], s=10, alpha=0.8, c="red", label="Fraud")
    plt.title("t-SNE Projection (Large Sample)")
    plt.legend()
    plt.show()

pca_tsne_plots(data.drop("Class", axis=1), data["Class"])